#                                                            Homework 2


Name : Luqman Khan

SID : S2024266054

Course: CS4152  Deep Learning & Neural Networks

Submitted to : Dr. Jameel Ahmad


# Question 1 : Concept-Based Training of an MLP in PyTorch

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)

model = nn.Sequential(
    nn.Linear(3, 4), nn.ReLU(),
    nn.Linear(4, 3), nn.Sigmoid(),
    nn.Linear(3, 1),
)


In [2]:
with torch.no_grad():
    model[0].weight.copy_(torch.tensor([[0.1, 0.2, -0.3], [0.4, -0.1, 0.2],
                                        [-0.2, 0.3, 0.1], [0.1, 0.1, -0.1]]))
    model[0].bias.copy_(torch.tensor([0.1, -0.1, 0.2, 0.0]))
    model[2].weight.copy_(torch.tensor([[0.2, -0.1, 0.3, 0.1], [0.1, 0.2, -0.2, 0.3],
                                        [-0.3, 0.1, 0.2, 0.2]]))
    model[2].bias.copy_(torch.tensor([0.0, 0.1, -0.1]))
    model[4].weight.copy_(torch.tensor([[0.5, -0.4, 0.3]]))
    model[4].bias.copy_(torch.tensor([0.1]))

x = torch.tensor([[1.0, 0.5, -1.0]])
y = torch.tensor([[1.5]])

# part(a) forward pass, layer by layer

In [3]:
z1 = model[0](x); a1 = model[1](z1)
z2 = model[2](a1); a2 = model[3](z2)
y_hat = model[4](a2)
loss = 0.5 * ((y - y_hat) ** 2).sum()
torch.set_printoptions(precision=6, sci_mode=False)
print("z1 =", z1.detach().squeeze()); print("a1 =", a1.detach().squeeze())
print("z2 =", z2.detach().squeeze()); print("a2 =", a2.detach().squeeze())
print(f"y_hat = {y_hat.item():.6f}   L = {loss.item():.6f}")

z1 = tensor([0.600000, 0.050000, 0.050000, 0.250000])
a1 = tensor([0.600000, 0.050000, 0.050000, 0.250000])
z2 = tensor([ 0.155000,  0.235000, -0.215000])
a2 = tensor([0.538673, 0.558481, 0.446456])
y_hat = 0.279881   L = 0.744346


# Part(b) autograd gradients + one SGD step on the OUTPUT layer only

In [4]:
opt = torch.optim.SGD(model[4].parameters(), lr=0.1)
opt.zero_grad()
loss.backward()
print("dL/dW3 =", model[4].weight.grad.squeeze())
print(f"dL/db3 = {model[4].bias.grad.item():.6f}")
opt.step()
print("W3_new =", model[4].weight.detach().squeeze())
print(f"b3_new = {model[4].bias.item():.6f}")

with torch.no_grad():
    y_new = model(x)
    loss_new = 0.5 * ((y - y_new) ** 2).sum()
print(f"y_hat_new = {y_new.item():.6f}   L_new = {loss_new.item():.6f}")


dL/dW3 = tensor([-0.657245, -0.681414, -0.544730])
dL/db3 = -1.220119
W3_new = tensor([ 0.565724, -0.331859,  0.354473])
b3_new = 0.222012
y_hat_new = 0.499672   L_new = 0.500328


# (d) Initialization Strategy
# (1) :
# For $W^{(1)}$ (followed by ReLU):Recommended:

He (Kaiming) Normal Initialization.   Formula: $\sigma = \sqrt{\frac{2}{n_{\text{in}}}} = \sqrt{\frac{2}{3}} \approx \mathbf{0.8165}$.Justification: ReLU sets negative inputs to zero, effectively deactivating roughly half the units. He initialization accounts for this by scaling variances by a factor of 2, preventing variance collapse during forward propagation.
# (2) : For $W^{(2)}$ (followed by Sigmoid):

 Recommended: Xavier (Glorot) Normal Initialization.   Formula: $\sigma = \sqrt{\frac{2}{n_{\text{in}} + n_{\text{out}}}} = \sqrt{\frac{2}{4 + 3}} = \sqrt{\frac{2}{7}} \approx \mathbf{0.5345}$ (or standard Xavier $\sigma = \sqrt{\frac{1}{n_{\text{in}}}} = \sqrt{\frac{1}{4}} = 0.5000$).
 **Justification:** Sigmoid saturates when inputs have large absolute values (causing vanishing gradients). Xavier maintains equal activation and gradient variances assuming a locally linear activation around zero.

##  (3): Why initializing all weights to zero fails (Symmetry Problem):

 ->  If all weights are initialized to 0, every neuron in a given hidden layer receives identical input ($z_i = 0$), produces identical activations ($a_i = g(0)$), and receives identical backpropagated error gradients.

 ->  Consequently, every neuron updates to the exact same value in every epoch, destroying network capacity and preventing neurons from learning distinct feature representations.